# Анализ конкурентов: пансионаты для пожилых (Москва и область)
**Студенческий проект | ВШЭ**  
Источники данных: 2GIS, Яндекс Карты, BookPansion  


In [ ]:
# 0. Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
from collections import Counter

# Настройка стиля
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.dpi'] = 120
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
sns.set_palette(PALETTE)
print("Библиотеки загружены ✓")

## 1. Загрузка данных

In [ ]:
# Укажи путь к папке с данными
DATA_PATH = "data/"   # <-- измени при необходимости

gis = pd.read_csv(DATA_PATH + "2gis_pansionaty_with_reviews.csv")
book = pd.read_csv(DATA_PATH + "book_pansionaty.csv")
yandex = pd.read_csv(DATA_PATH + "yandex_pansionaty.csv")

print(f"2GIS:       {gis.shape[0]} строк, {gis.shape[1]} колонок")
print(f"BookPansion:{book.shape[0]} строк, {book.shape[1]} колонок")
print(f"Яндекс:     {yandex.shape[0]} строк, {yandex.shape[1]} колонок")

## 2. Предобработка данных

In [ ]:
# --- 2GIS: оставляем нужные колонки, переименовываем ---
gis = gis.rename(columns={
    "name": "название",
    "rating": "рейтинг",
    "reviews": "число_отзывов",
    "lat": "широта",
    "lon": "долгота",
    "положительных_слов": "позитив_слов",
    "отрицательных_слов": "негатив_слов"
})

# --- BookPansion: парсим минимальную цену ---
book["min_price_per_day"] = pd.to_numeric(book["min_price_per_day"], errors="coerce")

# --- Яндекс: извлекаем минимальную цену из строки вида "Услуга: 1200; Услуга2: 1500" ---
def extract_min_price(s):
    if pd.isna(s):
        return None
    nums = re.findall(r':\s*(\d+)', str(s))
    if not nums:
        return None
    val = min(int(x) for x in nums)
    return val if val >= 500 else None  # фильтруем явные выбросы (разовые услуги < 500 ₽/день)

yandex["min_price"] = yandex["цены"].apply(extract_min_price)

# --- Яндекс: тип участника рынка (сеть / одиночка) ---
yandex["тип"] = yandex["кол-во по сети"].apply(lambda x: "Сеть" if x > 1 else "Одиночный")

print("Предобработка завершена ✓")
gis.head(3)

## 3. Обзор рынка: базовая статистика

In [ ]:
summary = pd.DataFrame({
    "Источник":         ["2GIS", "Яндекс", "BookPansion"],
    "Объектов":         [len(gis), len(yandex), len(book)],
    "Средний рейтинг":  [
        round(gis["рейтинг"].mean(), 2),
        round(yandex["рейтинг"].mean(), 2),
        round(book["rating"].mean(), 2)
    ],
    "Среднее отзывов":  [
        round(gis["число_отзывов"].mean(), 1),
        round(yandex["число отзывов"].mean(), 1),
        round(book["reviews"].mean(), 1)
    ],
})
summary

## График 1. Распределение рейтингов по источникам

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
datasets = [
    (gis["рейтинг"].dropna(),       "2GIS",       PALETTE[0]),
    (yandex["рейтинг"].dropna(),    "Яндекс",     PALETTE[1]),
    (book["rating"].dropna(),        "BookPansion", PALETTE[2]),
]

for ax, (data, title, color) in zip(axes, datasets):
    ax.hist(data, bins=10, color=color, edgecolor="white", linewidth=0.8, alpha=0.9)
    ax.axvline(data.mean(), color="black", linestyle="--", linewidth=1.2, label=f"Среднее: {data.mean():.2f}")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Рейтинг")
    ax.legend(fontsize=9)

axes[0].set_ylabel("Количество пансионатов")
fig.suptitle("Распределение рейтингов", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("plot1_rating_distribution.png", bbox_inches="tight")
plt.show()

print("Вывод: рейтинги в BookPansion выше и сосредоточены у 4.7–5.0 — агрегатор отбирает только лучших.")
print("Яндекс показывает реальный рынок с бимодальным распределением: много как высоких, так и низких оценок.")

## График 2. Рейтинг vs количество отзывов

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, r_col, n_col, label, color) in zip(axes, [
    (gis,    "рейтинг",  "число_отзывов",   "2GIS",   PALETTE[0]),
    (yandex, "рейтинг",  "число отзывов",   "Яндекс", PALETTE[1]),
]):
    d = df[[r_col, n_col]].dropna()
    ax.scatter(d[n_col], d[r_col], color=color, alpha=0.65, s=60, edgecolors="white", linewidth=0.5)
    # линия тренда
    z = np.polyfit(d[n_col], d[r_col], 1)
    p = np.poly1d(z)
    x_line = np.linspace(d[n_col].min(), d[n_col].max(), 100)
    ax.plot(x_line, p(x_line), color="black", linestyle="--", linewidth=1)
    ax.set_title(label, fontsize=13, fontweight="bold")
    ax.set_xlabel("Число отзывов")
    ax.set_ylabel("Рейтинг")

fig.suptitle("Рейтинг vs количество отзывов", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("plot2_rating_vs_reviews.png", bbox_inches="tight")
plt.show()

print("Вывод: слабая положительная связь — больше отзывов не гарантирует высокий рейтинг.")
print("Объекты в правом нижнем углу (много отзывов, низкий рейтинг) — потенциально слабые игроки с высокой известностью.")

## График 3. Топ-15 конкурентов по Яндекс (рейтинг × число отзывов)

In [ ]:
# Составной балл: рейтинг * log(число отзывов + 1)
yandex["score"] = yandex["рейтинг"] * np.log1p(yandex["число отзывов"])
top15 = yandex.nlargest(15, "score")[["название", "рейтинг", "число отзывов", "score"]].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top15["название"][::-1], top15["score"][::-1], color=PALETTE[0], alpha=0.85)

for bar, rating, reviews in zip(bars, top15["рейтинг"][::-1], top15["число отзывов"][::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f"★{rating} | {int(reviews)} отз.", va="center", fontsize=8.5, color="#333")

ax.set_xlabel("Составной балл (рейтинг × ln(отзывы+1))")
ax.set_title("Топ-15 пансионатов на Яндексе\n(сила позиции: рейтинг + активность отзывов)", fontsize=13, fontweight="bold")
ax.set_xlim(0, top15["score"].max() * 1.25)
plt.tight_layout()
plt.savefig("plot3_top15_yandex.png", bbox_inches="tight")
plt.show()

## График 4. Тональность отзывов (позитив / негатив)

In [ ]:
# Берём только тех, у кого есть хоть какая-то тональность
gis_sent = gis[(gis["позитив_слов"] > 0) | (gis["негатив_слов"] > 0)].copy()
yan_sent = yandex[(yandex["положительных_слов"] > 0) | (yandex["отрицательных_слов"] > 0)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, pos_col, neg_col, title, color_p, color_n in [
    (axes[0], gis_sent,  "позитив_слов",      "негатив_слов",        "2GIS",   "#55A868", "#C44E52"),
    (axes[1], yan_sent,  "положительных_слов", "отрицательных_слов",  "Яндекс", "#55A868", "#C44E52"),
]:
    top = df.nlargest(12, pos_col)[["название", pos_col, neg_col]].reset_index(drop=True)
    x = range(len(top))
    width = 0.38
    ax.bar([i - width/2 for i in x], top[pos_col], width, label="Позитивных слов", color=color_p, alpha=0.85)
    ax.bar([i + width/2 for i in x], top[neg_col], width, label="Негативных слов",  color=color_n, alpha=0.85)
    ax.set_xticks(list(x))
    ax.set_xticklabels(top["название"], rotation=40, ha="right", fontsize=8)
    ax.set_ylabel("Кол-во слов в отзывах")
    ax.set_title(f"Тональность отзывов — {title}", fontsize=12, fontweight="bold")
    ax.legend()

plt.tight_layout()
plt.savefig("plot4_sentiment.png", bbox_inches="tight")
plt.show()

print("Вывод: большинство пансионатов имеют явный перевес позитивных слов.")
print("Объекты с высоким негативом при низком позитиве — зоны риска для репутации.")

## График 5. Ценовой диапазон рынка (₽/день)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BookPansion — точечно
book_p = book["min_price_per_day"].dropna()
axes[0].hist(book_p, bins=8, color=PALETTE[2], edgecolor="white", alpha=0.9)
axes[0].axvline(book_p.mean(), linestyle="--", color="black", label=f"Среднее: {book_p.mean():.0f} ₽")
axes[0].axvline(book_p.median(), linestyle=":", color="gray", label=f"Медиана: {book_p.median():.0f} ₽")
axes[0].set_title("BookPansion — мин. цена/день", fontsize=12, fontweight="bold")
axes[0].set_xlabel("₽ / день")
axes[0].set_ylabel("Кол-во пансионатов")
axes[0].legend()

# Яндекс — min price parsed
yan_p = yandex["min_price"].dropna()
axes[1].hist(yan_p[yan_p <= 5000], bins=12, color=PALETTE[1], edgecolor="white", alpha=0.9)
axes[1].axvline(yan_p.mean(), linestyle="--", color="black", label=f"Среднее: {yan_p.mean():.0f} ₽")
axes[1].axvline(yan_p.median(), linestyle=":", color="gray", label=f"Медиана: {yan_p.median():.0f} ₽")
axes[1].set_title("Яндекс — мин. цена услуги/день", fontsize=12, fontweight="bold")
axes[1].set_xlabel("₽ / день")
axes[1].legend()

fig.suptitle("Ценовой диапазон рынка пансионатов", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plot5_prices.png", bbox_inches="tight")
plt.show()

print(f"Медианная минимальная цена по BookPansion: {book_p.median():.0f} ₽/день")
print(f"Медианная цена по Яндексу: {yan_p.median():.0f} ₽/день")

## График 6. Популярность услуг (BookPansion)

In [ ]:
services_all = []
for s in book["services"].dropna():
    services_all.extend([x.strip() for x in s.split("|")])

freq = Counter(services_all).most_common(12)
labels, counts = zip(*freq)

# Нормируем на кол-во пансионатов
total = len(book)
pct = [c / total * 100 for c in counts]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels[::-1], pct[::-1], color=PALETTE[0], alpha=0.85)
for bar, c in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.0f}%", va="center", fontsize=9)
ax.set_xlabel("Доля пансионатов, предлагающих услугу (%)")
ax.set_title("Топ-12 услуг пансионатов (BookPansion)", fontsize=13, fontweight="bold")
ax.set_xlim(0, 115)
plt.tight_layout()
plt.savefig("plot6_services.png", bbox_inches="tight")
plt.show()

print("Вывод: круглосуточный уход, 5-разовое питание и прогулки — стандарт рынка.")
print("Психолог и ЛФК — дифференцирующие услуги (есть не везде).")

## График 7. Сети vs одиночные пансионаты (Яндекс)

In [ ]:
network_counts = yandex["тип"].value_counts()
avg_rating = yandex.groupby("тип")["рейтинг"].mean()
avg_reviews = yandex.groupby("тип")["число отзывов"].mean()

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Pie
axes[0].pie(network_counts.values, labels=network_counts.index,
            colors=[PALETTE[0], PALETTE[2]], autopct="%1.0f%%",
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[0].set_title("Доля на рынке", fontsize=12, fontweight="bold")

# Avg rating
axes[1].bar(avg_rating.index, avg_rating.values, color=[PALETTE[0], PALETTE[2]], alpha=0.85, edgecolor="white")
for i, v in enumerate(avg_rating.values):
    axes[1].text(i, v + 0.05, f"{v:.2f}", ha="center", fontsize=11, fontweight="bold")
axes[1].set_title("Средний рейтинг", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Рейтинг")
axes[1].set_ylim(0, 5.5)

# Avg reviews
axes[2].bar(avg_reviews.index, avg_reviews.values, color=[PALETTE[0], PALETTE[2]], alpha=0.85, edgecolor="white")
for i, v in enumerate(avg_reviews.values):
    axes[2].text(i, v + 0.5, f"{v:.0f}", ha="center", fontsize=11, fontweight="bold")
axes[2].set_title("Среднее число отзывов", fontsize=12, fontweight="bold")
axes[2].set_ylabel("Отзывов")

fig.suptitle("Сетевые vs одиночные пансионаты", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plot7_network_vs_solo.png", bbox_inches="tight")
plt.show()

print("Вывод: сетевые пансионаты собирают больше отзывов (выше узнаваемость),")
print("но средний рейтинг у одиночных часто не ниже — качество не зависит от масштаба.")

## График 8. Карта конкурентов (2GIS)

In [ ]:
gis_map = gis.dropna(subset=["широта", "долгота", "рейтинг"])

fig, ax = plt.subplots(figsize=(9, 8))
sc = ax.scatter(
    gis_map["долгота"], gis_map["широта"],
    c=gis_map["рейтинг"], cmap="RdYlGn",
    vmin=1, vmax=5, s=80, alpha=0.85, edgecolors="white", linewidth=0.5
)

# Подписи топ-5 по числу отзывов
top5 = gis_map.nlargest(5, "число_отзывов")
for _, row in top5.iterrows():
    ax.annotate(row["название"][:25], (row["долгота"], row["широта"]),
                fontsize=7, ha="left",
                xytext=(5, 5), textcoords="offset points",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

cbar = plt.colorbar(sc, ax=ax, shrink=0.7)
cbar.set_label("Рейтинг", fontsize=10)
ax.set_xlabel("Долгота")
ax.set_ylabel("Широта")
ax.set_title("Карта пансионатов Москвы и области (2GIS)\nЦвет — рейтинг: красный = низкий, зелёный = высокий",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("plot8_map.png", bbox_inches="tight")
plt.show()

print("Вывод: большинство пансионатов сосредоточены в поясе вокруг МКАД.")
print("Объекты с низким рейтингом (красные) — потенциальные слабые места конкурентов.")

## График 9. Специализация пансионатов (BookPansion)

In [ ]:
spec_cols = {
    "dementia":   "Деменция",
    "alzheimer":  "Альцгеймер",
    "rehab":      "Реабилитация",
    "bedridden":  "Лежачие"
}

spec_counts = {v: book[k].sum() for k, v in spec_cols.items()}
total_book = len(book)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(spec_counts.keys(), spec_counts.values(), color=PALETTE, alpha=0.85, edgecolor="white")
for bar, count in zip(bars, spec_counts.values()):
    pct = count / total_book * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f"{count}\n({pct:.0f}%)", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("Кол-во пансионатов")
ax.set_title("Специализации пансионатов\n(доля из тех, кто указал специализацию)", fontsize=12, fontweight="bold")
ax.set_ylim(0, total_book + 2)
plt.tight_layout()
plt.savefig("plot9_specialization.png", bbox_inches="tight")
plt.show()

print("Вывод: уход за лежачими и реабилитация — наиболее распространённые специализации.")
print("Деменция и Альцгеймер требуют отдельных компетенций и позволяют занять нишу.")

## 10. Ключевые выводы

In [ ]:
conclusions = {
    "1. Рейтинг рынка":        "Медианный рейтинг ~4.4–4.9 — рынок в целом оценивается высоко, но хвост низких оценок (1–2★) значительный",
    "2. Ценовой диапазон":     "Медиана ~1 300–1 500 ₽/день; выбросы до 5 000 ₽/день — сегмент премиум существует",
    "3. Стандарт рынка":       "Круглосуточный уход + питание + прогулки есть у всех. Дифференциация — ЛФК, психолог, специализация",
    "4. Сети vs одиночки":     "31% — сетевые игроки. Более высокая узнаваемость, но не обязательно лучшее качество",
    "5. Репутационный риск":   "~20% пансионатов имеют негативные упоминания. Управление отзывами — ключевой фактор конкурентности",
    "6. Нишевые возможности":  "Деменция и Альцгеймер — специализированная ниша с меньшей конкуренцией и выше ценой",
}

for k, v in conclusions.items():
    print(f"\n{k}:\n  → {v}")